# 📝 벡터 검색 과제 LV1(기초) — 도서 소개 의미 검색

> 이 단원에서 배운 **임베딩**·**코사인 유사도 Top-K**·**키워드 검색과의 차이**·**ChromaDB(컬렉션·add·query·where 필터)** 를 **도서관 도서 소개** 데이터로 **한 문제에 하나씩** 확인하는 과제입니다.

## 풀이 방법
1. 맨 위 **제공 코드 셀**(라이브러리·임베딩 모델)을 먼저 실행하세요.
2. 각 문제의 **답안 셀**(`# 여기에 코드를 작성하세요`)에 코드를 채웁니다.
3. 바로 아래 **자가채점 셀**(`# [자가채점]`)을 실행해 `✅ 통과!` 가 뜨면 성공이에요.
4. 막히면 `힌트` 를 펼쳐 보세요.

> **자가채점이 없는 문제**: 서술형(1·12)입니다. 정답 노트북의 모범 서술과 비교하세요.

- 데이터는 `data/books.csv`(도서 17권, 열: `id`, `title` 제목, `genre` 장르, `description` 소개) 를 씁니다.
- 벡터 검색용 컬렉션 이름은 **`book_lib`** 로 통일합니다.

화이팅!

아래 셀을 먼저 실행해 라이브러리와 한국어 임베딩 모델을 준비하세요.

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한국어 임베딩 모델을 준비합니다.
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb

# 지난 단원에서 배운 한국어 임베딩 모델 — 문장 한 개를 768차원 벡터로 바꿉니다.
# (처음 부를 때 모델을 내려받느라 조금 걸릴 수 있어요.)
emb_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 준비 완료 — 벡터 차원:', emb_model.get_embedding_dimension())

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 어떤 데이터인지 파악합니다. (아래 셀은 실행만 하면 됩니다.)

In [ ]:
# [제공 코드] 도서 데이터를 살펴봅니다.
books = pd.read_csv('data/books.csv')
print("행·열 크기:", books.shape)
print("\n[장르별 개수]"); print(books["genre"].value_counts())
print("\n[앞 5행] head()"); display(books.head())

## 1. 데이터 살펴보기 (서술형)
**배경**: 분석 전에 데이터를 **눈으로 파악**하는 것이 첫걸음입니다. 위 셀 출력을 보고 이 도서 데이터에 대해 알게 된 사실을 정리해 보세요.

**요구사항**: 아래 서술 셀에 **관찰 2~3가지**를 문장으로 적으세요(도서가 몇 권인지, 장르 종류와 분포, 소개(`description`)가 어떤 내용인지 등).

> 이 문제는 자가채점이 없습니다. 정답 노트북의 모범 서술과 비교하세요.

*(여기에 관찰을 서술하세요)*

## 2. 문장 하나를 임베딩하기
**배경**: 의미 검색의 출발은 문장을 **768차원 벡터**로 바꾸는 것입니다. `emb_model.encode` 에 **문자열 리스트**를 넣고 `normalize_embeddings=True` 를 주면 코사인 검색에 바로 쓸 단위벡터가 나옵니다.

**요구사항**:
- 문장 `"모험을 떠나는 이야기"` 하나를 리스트에 담아 `emb_model.encode(..., normalize_embeddings=True)` 로 임베딩해 변수 `one_vec` 에 담으세요.
- `one_vec.shape` 가 `(1, 768)` 인지 확인하세요(문장 1개 × 768차원).

**예시**
```
one_vec.shape  →  (1, 768)
```
<details><summary>힌트</summary>

```text
접근방법:
- encode 는 문자열들의 리스트를 받아 (문장 수, 768) 모양의 행렬을 돌려준다.

세부구현:
1. 문장 하나를 대괄호로 감싼 리스트로 만든다
2. emb_model.encode 에 그 리스트와 normalize_embeddings=True 를 넘겨 one_vec 에 담는다
3. one_vec.shape 를 출력해 (1, 768) 인지 확인한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert one_vec.shape == (1, 768)
print("✅ 문제2 통과!")

## 3. 여러 문장을 한 번에 임베딩하기
**배경**: 문서 전체를 색인하려면 **여러 문장을 한 번에** 임베딩합니다. `description` 열 전체를 넣어 봅시다.

**요구사항**:
- `books['description'].tolist()` 로 소개 문장 리스트를 만들어 `emb_model.encode(..., normalize_embeddings=True)` 로 임베딩해 변수 `book_emb` 에 담으세요.
- `book_emb.shape` 가 `(17, 768)` 인지 확인하세요(도서 17권 × 768차원).

**예시**
```
book_emb.shape  →  (17, 768)
```
<details><summary>힌트</summary>

```text
접근방법:
- description 열을 리스트로 만들어 통째로 encode 에 넘긴다.

세부구현:
1. books 의 description 열을 tolist 로 리스트로 만든다
2. 그 리스트를 encode 에 normalize_embeddings=True 와 함께 넘겨 book_emb 에 담는다
3. book_emb.shape 로 (17, 768) 을 확인한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert book_emb.shape == (17, 768)
# 제목이 아니라 소개(description) 열을 임베딩했는지 — 첫 책으로 대조한다
_one = emb_model.encode([books.loc[0, 'description']], normalize_embeddings=True)[0]
assert np.allclose(book_emb[0], _one, atol=1e-3), \
    'description 열을 임베딩해야 합니다(title 이나 다른 열이면 값이 달라집니다)'
print("✅ 문제3 통과!")

## 4. 코사인 유사도 — 질문과 가장 비슷한 책 1권
**배경**: 이제 **질문 문장**과 가장 비슷한 책을 찾습니다. 질문을 임베딩해 문서 임베딩(`book_emb`)과 코사인 유사도를 구하고, 가장 높은 1권을 고릅니다.

**요구사항**:
- 문제 3 의 `book_emb` 를 사용합니다.
- 질문 `"무서운 살인 사건을 파헤치는 이야기"` 를 임베딩해 `cosine_similarity(질문벡터, book_emb)[0]` 로 유사도 배열(길이 17)을 구해 변수 `sims` 에 담으세요.
- 유사도가 가장 높은 책의 인덱스를 `best_idx` 에 정수로 담고, 그 책의 장르를 `best_genre = books.loc[best_idx, 'genre']` 로 구하세요.

**예시**
```
len(sims)   →  17
best_genre  →  '추리'   (살인 사건 이야기이므로 추리 장르가 가장 가깝다)
```
<details><summary>힌트</summary>

```text
접근방법:
- 질문 한 문장을 임베딩해 전체 문서와의 코사인 유사도를 구하고, 최댓값 위치를 찾는다.

세부구현:
1. 질문을 리스트로 감싸 encode 로 임베딩한다
2. cosine_similarity 에 (질문벡터, book_emb) 를 넘겨 첫 행을 sims 에 담는다
3. np.argmax 로 가장 높은 인덱스를 best_idx 에 담고, books.loc 로 그 책의 genre 를 얻는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(sims) == 17
assert best_genre == '추리'
# 실제로 유사도를 계산해 최댓값을 골랐는지 — 아무 추리 책 인덱스를 적어 넣으면 걸린다
_q = emb_model.encode(['무서운 살인 사건을 파헤치는 이야기'], normalize_embeddings=True)
_sims4 = cosine_similarity(_q, book_emb)[0]
assert np.allclose(sims, _sims4, atol=1e-3), 'sims 는 이 질문과의 코사인 유사도여야 합니다'
assert best_idx == int(np.argmax(sims)), 'best_idx 는 유사도가 가장 높은 책의 인덱스여야 합니다'
print("✅ 문제4 통과!")

## 5. 코사인 유사도 — 특정 책과 비슷한 책 Top-3
**배경**: 이번엔 질문 대신 **책 한 권**을 기준으로, 그 책과 가장 비슷한 다른 책 3권을 찾습니다(추천 시스템의 기본입니다). 자기 자신은 빼야 합니다.

**요구사항**:
- 기준 책은 **3번 인덱스**(`book_emb[3]`)입니다. `cosine_similarity(book_emb[3:4], book_emb)[0]` 로 그 책과 모든 책의 유사도를 구해 변수 `sims3` 에 담으세요.
- `np.argsort(-sims3)` 은 유사도 내림차순 인덱스입니다. **맨 앞(자기 자신 3번)을 빼고** 상위 3개를 변수 `top3` 에 담으세요.

**예시**
```
len(top3)     →  3
3 in top3     →  False   (자기 자신은 제외)
```
<details><summary>힌트</summary>

```text
접근방법:
- 기준 책 벡터와 전체의 코사인 유사도를 구하고, 내림차순 인덱스에서 맨 앞(자기 자신)을 건너뛴 3개를 고른다.

세부구현:
1. cosine_similarity 에 (book_emb 의 3번 한 행, 전체 book_emb) 을 넘겨 첫 행을 sims3 에 담는다
2. 유사도에 마이너스를 붙여 argsort 하면 내림차순 인덱스가 나온다
3. 그 인덱스의 1~3번(0번=자기자신 제외)을 top3 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(top3) == 3
assert 3 not in list(top3)
# 실제로 유사도로 정렬해 뽑았는지 — 임의의 인덱스([0,1,2] 등)로는 통과하지 못한다
_sims = cosine_similarity(book_emb[3:4], book_emb)[0]
assert list(top3) == list(np.argsort(-_sims)[1:4]), '유사도 내림차순 1~3위여야 합니다'
print("✅ 문제5 통과!")

## 6. ChromaDB 컬렉션 만들고 문서 넣기
**배경**: 이제 문서 벡터를 **벡터 DB** 에 저장합니다. 메모리 클라이언트를 열고, 코사인 거리의 컬렉션 `book_lib` 를 만들어 17권을 적재합니다.

**요구사항**:
- `chromadb.EphemeralClient()` 로 클라이언트를 열어 변수 `client` 에 담으세요.
- `client.get_or_create_collection('book_lib', metadata={'hnsw:space': 'cosine'})` 로 컬렉션을 만들어 변수 `book_lib` 에 담으세요.
- `book_lib.add(...)` 로 문서를 넣으세요 — `ids=books['id'].tolist()`, `embeddings=book_emb`, `documents=books['description'].tolist()`, `metadatas` 는 각 책의 `{'title':…, 'genre':…}` 리스트.
- `book_lib.count()` 가 `17` 인지 확인하세요.

**예시**
```
book_lib.count()  →  17
```
<details><summary>힌트</summary>

```text
접근방법:
- 클라이언트를 열고, 코사인 컬렉션을 만든 뒤, id·임베딩·원문·메타데이터를 함께 add 한다.

세부구현:
1. EphemeralClient 로 client 를 만든다
2. get_or_create_collection 으로 book_lib 을 만든다(metadata 에 hnsw:space=cosine)
3. add 에 ids·embeddings·documents·metadatas 를 넘긴다(메타는 title·genre 딕셔너리 리스트)
4. count 로 17 을 확인한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert book_lib.count() == 17
print("✅ 문제6 통과!")

아래는 이후 검색 문제(7~10)에서 쓸 `book_lib` 컬렉션을 **확실히 준비**하는 제공 코드입니다. (문제 6을 이미 풀었다면 그대로 유지되고, 건너뛰었더라도 여기서 채워집니다.)

In [ ]:
# [제공 코드] 검색 문제에서 쓸 컬렉션(book_lib)을 준비합니다.
# 이미 앞에서 만들었다면 그대로 유지되고, 안 만들었더라도 여기서 채워집니다.
books = pd.read_csv('data/books.csv')
book_emb = emb_model.encode(books['description'].tolist(), normalize_embeddings=True)
client = chromadb.EphemeralClient()
book_lib = client.get_or_create_collection('book_lib', metadata={'hnsw:space': 'cosine'})
book_lib.add(
    ids=books['id'].tolist(),
    embeddings=book_emb,
    documents=books['description'].tolist(),
    metadatas=[{'title': t, 'genre': g} for t, g in zip(books['title'], books['genre'])],
)
print('book_lib 문서 수:', book_lib.count())

## 7. 컬렉션에 질문하기 — Top-3 검색
**배경**: 이제 코사인 계산을 손으로 하지 않고 `book_lib.query(...)` 한 줄로 검색합니다.

**요구사항**:
- 질문 `"아이에게 읽어 줄 동물이 나오는 이야기책"` 을 임베딩해 변수 `q_emb` 에 담으세요.
- `book_lib.query(query_embeddings=q_emb, n_results=3)` 를 호출해 결과를 `res` 에 담으세요.
- 가까운 문서 id 리스트를 `hit_ids = res['ids'][0]` 로 꺼내세요.
- 이 질문에는 동화 `"숲속 동물 재판"`(id `b08`)이 가장 잘 맞습니다. `'b08'` 이 `hit_ids` 에 들어 있는지 확인합니다.

**예시**
```
len(hit_ids)      →  3
'b08' in hit_ids  →  True
```
<details><summary>힌트</summary>

```text
접근방법:
- 질문을 임베딩해 query_embeddings 로 넣고 n_results=3 으로 검색한 뒤, ids 의 0번(첫 질문 결과)을 꺼낸다.

세부구현:
1. 질문을 리스트로 감싸 encode 로 q_emb 에 담는다
2. book_lib.query 에 query_embeddings=q_emb, n_results=3 을 넘겨 res 에 담는다
3. res['ids'][0] 을 hit_ids 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(hit_ids) == 3
assert 'b08' in hit_ids
print("✅ 문제7 통과!")

## 8. 다른 질문으로 Top-3 검색
**배경**: 같은 `query` 를 **다른 질문**에 적용해 봅니다.

**요구사항**:
- 질문 `"돈을 모으고 투자하는 법을 배우고 싶다"` 를 임베딩해 `book_lib.query(..., n_results=3)` 로 검색하고, 문서 id 리스트를 `money_ids` 에 담으세요.
- 이 질문에는 경제서 `"돈의 흐름을 읽는 법"`(id `b04`)이 가장 잘 맞습니다. `'b04'` 가 `money_ids` 에 들어 있는지 확인합니다.

**예시**
```
'b04' in money_ids  →  True
```
<details><summary>힌트</summary>

```text
접근방법:
- 문제 7 과 같은 흐름을 질문만 바꿔 반복하고, ids 의 0번을 꺼낸다.

세부구현:
1. 질문을 임베딩한다
2. book_lib.query 로 n_results=3 검색한다
3. 결과의 ids[0] 을 money_ids 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert len(money_ids) == 3, 'n_results=3 으로 검색해야 합니다'
assert 'b04' in money_ids
print("✅ 문제8 통과!")

## 9. 키워드 검색과 의미 검색은 무엇이 다를까
**배경**: 우리가 흔히 쓰는 **키워드 검색**은 질문의 **낱말이 문서에 그대로 있는지**만 봅니다. 반면 지금까지 해 온 **의미 검색**은 낱말이 겹치지 않아도 **뜻이 가까우면** 찾아 줍니다. 같은 주제로 두 방식을 나란히 돌려 그 차이를 직접 확인해 봅시다.

**요구사항**:
- **키워드 검색**: `books['description'].str.contains('요리')` 로 소개에 '요리'라는 낱말이 **그대로** 들어간 책만 골라, 그 책들의 `id` 리스트를 `kw_ids` 에 담으세요.
- **의미 검색**: 질문 `"요리를 배우고 싶어요"` 를 임베딩해 `book_lib.query(..., n_results=3)` 로 검색하고, 문서 id 리스트를 `sem_ids` 에 담으세요.
- 두 결과를 나란히 출력해, **키워드 검색이 놓친 책을 의미 검색이 찾아냈는지** 확인하세요.

**예시**
```
kw_ids   →  ['b05']                  (소개에 '요리'라는 낱말이 든 책은 한 권뿐)
sem_ids  →  ['b05', 'b10', 'b15']    (요리책 세 권을 모두 찾아냄)
```
<details><summary>힌트</summary>

```text
접근방법:
- 낱말 매칭은 판다스 문자열 필터로, 의미 검색은 지금까지 쓰던 query 로 각각 구해 비교한다.

세부구현:
1. books 의 description 에 str.contains 로 '요리' 가 든 행만 남기고, 그 id 를 리스트로 만든다
2. 질문을 임베딩해 book_lib.query 로 n_results=3 검색한다
3. 두 id 리스트를 출력하고, 의미 검색에만 있는 id 를 확인한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert kw_ids == ['b05'], "소개에 '요리'라는 낱말이 그대로 든 책은 b05 한 권뿐입니다"
assert len(sem_ids) == 3, 'n_results=3 으로 검색해야 합니다'
assert 'b10' in sem_ids and 'b15' in sem_ids, \
    "의미 검색은 소개에 '요리'가 없는 요리책(b10 발효음식·b15 베이킹)도 찾아냅니다"
# 두 방식의 결과가 실제로 달라야 한다 — 의미 검색이 키워드가 놓친 책을 더 찾는다
assert set(sem_ids) - set(kw_ids), '의미 검색에만 있는 책이 있어야 합니다'
print('✅ 문제9 통과! 키워드', len(kw_ids), '권 → 의미 검색', len(sem_ids), '권')

## 10. 메타데이터 필터 — 추리 장르 안에서만
**배경**: 의미 검색에 **장르 조건**을 함께 걸어 봅니다. `query` 의 `where` 인자에 `{'메타데이터 열': 값}` 형태의 조건을 주면, 그 조건에 맞는 문서만 검색 대상이 됩니다.

**요구사항**:
- 질문 `"긴장감 넘치는 이야기"` 를 임베딩해 `book_lib.query(..., n_results=5)` 로 검색하되, **장르가 '추리' 인 책만** 나오도록 `where` 조건을 함께 거세요. 결과는 `res` 에 담습니다.
- 나온 문서들의 장르 리스트를 `genres = [m['genre'] for m in res['metadatas'][0]]` 로 만드세요.
- `genres` 가 **모두 '추리'** 인지 확인합니다(추리는 4권이라 5개를 요청해도 4개가 나옵니다).

**예시**
```
genres  →  ['추리', '추리', '추리', '추리']   (모두 추리)
```
<details><summary>힌트</summary>

```text
접근방법:
- query 에 where 로 장르 조건을 함께 넘긴다. 반환된 메타데이터에서 genre 만 뽑아 확인한다.

세부구현:
1. 질문을 임베딩한다
2. book_lib.query 에 n_results=5 와 함께, 장르(genre)를 '추리' 로 고정하는 where 조건을 넘긴다
3. res['metadatas'][0] 의 각 원소에서 genre 를 뽑아 genres 리스트를 만든다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# n_results=5 를 지켰는지 — 추리는 4권이라 정확히 4개가 나와야 한다(1로 줄여 풀면 걸린다)
assert len(genres) == 4, 'n_results=5 로 요청하면 추리 4권이 모두 나옵니다'
assert all(g == '추리' for g in genres)
print("✅ 문제10 통과!")

## 11. 메타데이터 필터 — 동화 장르 안에서만
**배경**: 같은 필터를 **다른 장르**에 적용합니다.

**요구사항**:
- 질문 `"따뜻한 이야기"` 를 임베딩해 `book_lib.query(..., n_results=5)` 로 검색하되, 이번엔 **장르가 '동화' 인 책만** 나오도록 `where` 조건을 거세요. 나온 문서들의 장르 리스트를 `dong_genres` 에 담으세요.
- `dong_genres` 가 **모두 '동화'** 인지 확인합니다.

**예시**
```
dong_genres  →  ['동화', '동화', '동화']   (모두 동화)
```
<details><summary>힌트</summary>

```text
접근방법:
- 문제 10 과 같은 흐름에서 where 의 장르만 '동화' 로 바꾼다.

세부구현:
1. 질문을 임베딩한다
2. book_lib.query 에 n_results=5 와 함께, 장르를 '동화' 로 고정하는 where 조건을 넘긴다
3. 메타데이터에서 genre 를 뽑아 dong_genres 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
# 동화는 3권 — 요청 개수를 지켰는지 정확값으로 확인한다
assert len(dong_genres) == 3, '동화 3권이 모두 나와야 합니다'
assert all(g == '동화' for g in dong_genres)
print("✅ 문제11 통과!")

## 12. 벡터 DB 솔루션 비교 (서술형)
**배경**: 교안에서 **Pinecone·ChromaDB·Qdrant** 세 벡터 DB 를 비교했습니다. 각각의 성격을 여러분의 말로 정리해 보세요.

**요구사항**: 아래 서술 셀에 세 솔루션의 **한 줄 특징**을 각각 적으세요(로컬/클라우드, 가벼움/프로덕션 성능 등). 그리고 '이 과정에서 기본으로 쓰는 것은 무엇이고 왜인지'도 한 문장 덧붙이세요.

> 이 문제는 자가채점이 없습니다. 정답 노트북의 모범 서술과 비교하세요.

*(여기에 세 솔루션의 특징과 기본 선택을 서술하세요)*